# 10 · Layer 3：Graph Workflows（ADK 2.0 的執行引擎）

前面兩章留下一個沒解決的缺口：

> `SequentialAgent` / `ParallelAgent` / `LoopAgent` **做不到條件分支**。

「金額超過一萬就要主管簽核，否則直接放行」這種再普通不過的需求，
三個 workflow agent 都表達不出來。

ADK 2.0 把整個執行核心換成了**圖形化引擎**，補上了這一塊。

```
       START
         │
         ▼
     ┌────────┐
     │ 分類   │
     └───┬────┘
         │  ctx.route = "urgent" / "normal"
    ┌────┴────┐
    ▼         ▼
 緊急處理   一般處理
```

## 0. 環境

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

## 1. 其實你一直都在用圖

ADK 2.0 之後，**連跑一個單一 agent 都會走圖形引擎**。
不信的話，看看單一 agent 出錯時的 traceback：

In [2]:
from google.adk.agents import LlmAgent

broken = LlmAgent(name="broken", model="gemini-does-not-exist", instruction="hi")

try:
    await run_once(broken, "hello")
except Exception as exc:
    import traceback

    frames = traceback.extract_tb(exc.__traceback__)
    workflow_frames = [f for f in frames if "workflow" in f.filename]
    print(f"例外類型: {type(exc).__name__}")
    print("\ntraceback 裡跟 workflow 引擎有關的層:")
    for f in workflow_frames[:5]:
        print(f"  {f.filename.split('site-packages/')[-1]}:{f.lineno} in {f.name}")

例外類型: ClientError

traceback 裡跟 workflow 引擎有關的層:
  google/adk/workflow/_node_runner.py:136 in run
  google/adk/workflow/_node_runner.py:274 in _execute_node
  google/adk/workflow/_node_runner.py:288 in _run_node_loop
  google/adk/workflow/_base_node.py:170 in run
  google/adk/workflow/_llm_agent_wrapper.py:484 in run_llm_agent_as_node


一個孤零零的 `LlmAgent` 也會經過 `workflow/_node_runner.py`，
而且失敗時拋的是 `DynamicNodeFailError`——**每個 agent 都是圖上的一個節點**。

這就是 ADK 2.0 最大的架構改變：從「階層式執行器」換成「圖形化執行引擎」。

## 2. 最小的圖

三個要素：

| 概念 | 是什麼 |
|---|---|
| **節點（node）** | 一個 `LlmAgent`，或一個用 `@node` 裝飾的函式 |
| **邊（edge）** | `(從哪, 到哪)` 的 tuple |
| **`START`** | 圖的入口，使用者訊息從這裡進來 |

In [3]:
from google.adk import Workflow
from google.adk.workflow import START, node


@node
def shout(text: str = "") -> str:
    return f"你說的是：「{text}」"


greeting = LlmAgent(
    name="greeting",
    model=get_model(),
    instruction="用一句繁體中文歡迎使用者。",
    output_key="text",
)

simple = Workflow(
    name="simple_flow",
    edges=[
        (START, greeting),
        (greeting, shout),
    ],
)
print("節點數（邊）:", len(simple.edges))

節點數（邊）: 2


### 圖的輸出在 `event.output`，不在 `event.content`

這是從 workflow agent 換過來時最容易卡住的地方。
`@node` 函式的回傳值放在 **`event.output`**，
而 `is_final_response()` 抓的是 LLM 的文字回應——兩者是不同的東西。

In [4]:
from google.adk.runners import InMemoryRunner
from google.genai import types


async def run_graph(workflow, text: str):
    """跑一張圖，把每個節點的 output 印出來，並回傳最後一個。"""
    runner = InMemoryRunner(agent=workflow, app_name="concept_track")
    sid = await new_session(runner)
    message = types.Content(role="user", parts=[types.Part(text=text)])
    last = None
    async for event in runner.run_async(
        user_id="student", session_id=sid, new_message=message
    ):
        output = getattr(event, "output", None)
        if output is not None:
            print(f"  ▪ {output}")
            last = output
    return last


await run_graph(simple, "哈囉")

  ▪ 你說的是：「歡迎光臨！請問有什麼我可以協助您的嗎？」


'你說的是：「歡迎光臨！請問有什麼我可以協助您的嗎？」'

## 3. 節點的參數是從 **state** 綁進來的

`@node` 裝飾的函式，它的參數**預設從 session state 取值**，
不是從上一個節點的回傳值。

所以上面的 `shout(text)` 拿到的是 `greeting` 用 `output_key="text"` 寫進去的值。

這是一個很容易誤會的設計。忘記設 `output_key`，就會得到：

```
ValueError: Missing value for parameter "text" of function "shout".
            It was not found in state and has no default value.
```

> 想改成綁「上一個節點的輸出」，用 `@node(parameter_binding="node_input")`。

## 4. 條件分支：`ctx.route`

這是本章的重點，也是 workflow agent 做不到的事。

規則：

1. 分流節點多收一個 `ctx: Context` 參數
2. 在裡面設 `ctx.route = "某個值"`
3. 邊用 **dict** 表示：`(分流節點, {"值A": 節點A, "值B": 節點B})`

**注意**：路由靠的是 `ctx.route`，**不是函式的回傳值**。

In [5]:
from google.adk.agents.context import Context
from pydantic import BaseModel, Field


class Triage(BaseModel):
    urgent: bool = Field(description="是否為緊急事件")
    topic: str = Field(description="問題主題，五個字以內")


triage = LlmAgent(
    name="triage",
    model=get_model(),
    instruction="判斷使用者回報的問題是否緊急（系統中斷、資料遺失、影響多數使用者算緊急），並歸納主題。",
    output_schema=Triage,
    output_key="triage",
)


@node
def router(ctx: Context, triage: dict) -> str:
    ctx.route = "urgent" if triage["urgent"] else "normal"
    return f"分流結果 → {ctx.route}"


@node
def urgent_path(triage: dict) -> str:
    return f"[P1] {triage['topic']}：已開緊急工單、通知值班工程師"


@node
def normal_path(triage: dict) -> str:
    return f"[P3] {triage['topic']}：排入客服佇列，24 小時內回覆"


support_flow = Workflow(
    name="support_flow",
    edges=[
        (START, triage),
        (triage, router),
        (router, {"urgent": urgent_path, "normal": normal_path}),
    ],
)

In [6]:
print("=== 緊急案件 ===")
await run_graph(support_flow, "整個網站掛掉了，所有客戶都連不上！")

print("\n=== 一般案件 ===")
await run_graph(support_flow, "請問發票可以改成公司抬頭嗎？")

=== 緊急案件 ===


  ▪ 分流結果 → urgent
  ▪ [P1] 系統中斷：已開緊急工單、通知值班工程師

=== 一般案件 ===


  ▪ 分流結果 → normal
  ▪ [P3] 發票抬頭：排入客服佇列，24 小時內回覆


'[P3] 發票抬頭：排入客服佇列，24 小時內回覆'

**同一張圖，兩條路徑。** 這是 `SequentialAgent` 永遠做不到的事。

而且分支條件寫在 Python 裡（`triage["urgent"]`），是**確定性的**——
不像交棒那樣每次都要賭模型的判斷。模型只負責「判斷緊不緊急」這件它擅長的事，
「緊急的話要走哪條路」由程式決定。

## 5. Fan-out / Join：平行展開再合流

`JoinNode` 會等所有上游節點都跑完才觸發。

In [7]:
from google.adk.workflow import JoinNode

security = LlmAgent(
    name="security_review",
    model=get_model(),
    description="資安審查",
    instruction="從資安角度審查使用者提出的方案，指出最大的一個風險，一句話，繁體中文。",
    output_key="security",
)

legal = LlmAgent(
    name="legal_review",
    model=get_model(),
    description="法遵審查",
    instruction="從法遵角度審查使用者提出的方案，指出最大的一個疑慮，一句話，繁體中文。",
    output_key="legal",
)

cost = LlmAgent(
    name="cost_review",
    model=get_model(),
    description="成本審查",
    instruction="從成本角度審查使用者提出的方案，指出最大的一項支出，一句話，繁體中文。",
    output_key="cost",
)

gather = JoinNode(name="gather")


@node
def summarize(security: str, legal: str, cost: str) -> str:
    return (
        "三方審查結果：\n"
        f"  🔒 資安：{security}\n"
        f"  ⚖️  法遵：{legal}\n"
        f"  💰 成本：{cost}"
    )


review_flow = Workflow(
    name="review_flow",
    edges=[
        (START, (security, legal, cost)),   # fan-out：一次觸發三個
        (security, gather),
        (legal, gather),
        (cost, gather),                      # join：三個都到齊才往下
        (gather, summarize),
    ],
)

In [8]:
print(await run_graph(review_flow, "我們打算把客戶資料放到公有雲的向量資料庫做 RAG。"))

  ▪ {'cost_review': '此方案最大的一項支出是持續的公有雲端「資料傳輸與儲存費用」，以及隨著資料量增長而暴增的「向量檢索運算成本」。', 'legal_review': '本方案最大的法遵疑慮在於：將未經適當去識別化的客戶敏感資料存放於公有雲向量資料庫，恐違反《個人資料保護法》及相關資安法規關於資料安全維護與跨境傳輸的合規要求。', 'security_review': '最大風險在於**敏感客戶資料上傳至第三方公有雲後，若缺乏完善的端到端加密與嚴格的存取控制，將面臨資料外洩、遭到未授權存取或被用作AI模型訓練的嚴重合規危機**。'}
  ▪ 三方審查結果：
  🔒 資安：最大風險在於**敏感客戶資料上傳至第三方公有雲後，若缺乏完善的端到端加密與嚴格的存取控制，將面臨資料外洩、遭到未授權存取或被用作AI模型訓練的嚴重合規危機**。
  ⚖️  法遵：本方案最大的法遵疑慮在於：將未經適當去識別化的客戶敏感資料存放於公有雲向量資料庫，恐違反《個人資料保護法》及相關資安法規關於資料安全維護與跨境傳輸的合規要求。
  💰 成本：此方案最大的一項支出是持續的公有雲端「資料傳輸與儲存費用」，以及隨著資料量增長而暴增的「向量檢索運算成本」。
三方審查結果：
  🔒 資安：最大風險在於**敏感客戶資料上傳至第三方公有雲後，若缺乏完善的端到端加密與嚴格的存取控制，將面臨資料外洩、遭到未授權存取或被用作AI模型訓練的嚴重合規危機**。
  ⚖️  法遵：本方案最大的法遵疑慮在於：將未經適當去識別化的客戶敏感資料存放於公有雲向量資料庫，恐違反《個人資料保護法》及相關資安法規關於資料安全維護與跨境傳輸的合規要求。
  💰 成本：此方案最大的一項支出是持續的公有雲端「資料傳輸與儲存費用」，以及隨著資料量增長而暴增的「向量檢索運算成本」。


> **踩坑**：`JoinNode` 只會等**實際會被觸發**的上游。
> 如果某個分支因為條件路由沒被走到，join 會一直等下去（官方文件稱為
> *Stuck JoinNode*）。條件分支與 join 混用時要特別小心。

## 6. 圖 vs Workflow Agent

| | Workflow Agent | Graph Workflow |
|---|---|---|
| 條件分支 | ❌ | ✅ `ctx.route` |
| 平行 + 合流 | 只能整批平行 | ✅ `JoinNode` 細緻控制 |
| 資料傳遞 | `output_key` + `{key?}` | 節點參數從 state 綁定 |
| 輸出位置 | `event.content` | **`event.output`** |
| 迴圈 | `LoopAgent` | 邊可以指回前面的節點 |
| 版本狀態 | 2.8.0 起 deprecated | 2.0 起的主線 |

**兩者的資料流機制是不同的**，混用時要清楚自己在用哪一套。

## 7. 已知限制

動手前值得先知道的幾件事：

- **Stuck JoinNode**：等一個永遠不會來的上游。
- **一個節點一次執行只會發出一個 `Event.output`**。
- **state 不要放大東西**：整張圖共用，塞大檔案會拖垮效能——用 Artifact。
- **`Workflow` 目前還不能當 `LlmAgent` 的 sub_agent**，兩套編排無法任意巢狀。

## 本章重點

- **ADK 2.0 的執行核心就是圖**，連單一 agent 都是圖上的一個節點。
- **三個要素**：節點（agent 或 `@node` 函式）、邊（tuple）、`START`。
- **節點參數預設從 state 綁定**，所以上游 agent 要記得設 `output_key`。
- **條件分支靠 `ctx.route`**，不是回傳值；邊用 dict 表示分支。
  分支條件是確定性的 Python 程式碼，比交棒可靠。
- **圖的輸出在 `event.output`**，不是 `event.content`。
- **`JoinNode` 做 fan-out / join**，但小心 Stuck JoinNode。

## 動手練習

1. 幫 `support_flow` 加第三條路 `"spam"`，讓明顯是廣告的訊息直接被丟掉。
2. 把 `router` 的 `ctx.route` 那行改成 `return`（只回傳不設 route），
   重跑，確認分支不會發生。
3. 在 `review_flow` 的 `START` 前面加一個分流節點，讓「內部工具」類的方案
   跳過法遵審查——然後觀察 `JoinNode` 會不會卡住。

---
**下一站 → `11_capstone.ipynb`**：把十一章的東西組成一個完整專案，
並且離開 notebook、用 `adk web` 跑起來。